This notebook converts the Top Filtered sites into a dictionary for use in Keras.

In [1]:
import duckdb
import json

In [ ]:
# !pip install python-pptx


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\mroth\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install python-pptx

'c:\Users\mroth\Comp' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
con = duckdb.connect("../../data/database/database.duckdb", read_only=True)

df = con.execute("SELECT * FROM main_marts.lp3_filtered_sites").df()
con.close()

# Drop the column
df = df.drop(columns=["flood_severity_score"])

print(df.shape)
print(df.head())

(37, 7)
    site_id        Q2_cfs        Q5_cfs       Q10_cfs       Q25_cfs  \
0  06803500   8634.149248  14142.051066  18384.377264  24401.192153   
1  06446700    238.620694    566.577122    901.459275   1493.235085   
2  06793000  12484.448618  25450.153875  37639.812747  57975.445451   
3  06465700   2014.433081   6125.696296  11109.423350  21184.199458   
4  06799100   1278.664536   3804.952310   7671.027376  17996.148551   

        Q50_cfs       Q100_cfs  
0  29352.866334   34702.933697  
1   2079.637762    2811.346507  
2  77260.623764  100582.275167  
3  32333.308336   47483.913901  
4  33132.127459   59760.463042  


In [5]:
# Convert to dictionary and save to JSON
data_dict = df.set_index("site_id").to_dict(orient="index")

with open("top_site_quantile_thresholds.json", "w") as f:
    json.dump(data_dict, f, indent=4)

# Create a dictionary of upstream gauges to the primary gauges

In [6]:
site_ids = df["site_id"].tolist()
print(site_ids)

['06803500', '06446700', '06793000', '06465700', '06799100', '06917000', '06820500', '06838000', '06803495', '06700000', '06449000', '06036650', '06710385', '06440200', '06207500', '06447500', '06784000', '06892000', '06464100', '06188000', '06821500', '06803513', '06061500', '06935955', '06895000', '06799315', '06881000', '06935755', '06921590', '06917060', '06211000', '06869950', '06308500', '06093200', '06450500', '06909500', '06834000']


In [21]:
upstream_pair_dict = {'06803500': '06803495',
                        '06446700': '06447500',
                        '06793000': '06790500',
                        '06465700': None,
                        '06799100': None,
                        '06917000': None,
                        '06820500': '06820410',
                        '06838000': None,
                        '06803495': '06803486',
                        '06700000': None,
                        '06449000': None,
                        '06036650': '06027600',
                        '06710385': None,
                        '06440200': None,
                        '06207500': '06209500',
                        '06447500': None,
                        '06784000': None,
                        '06892000': None,
                        '06464100': None,
                        '06188000': None,
                        '06821500': None,
                        '06803513': '06803500',
                        '06061500': '06062500',
                        '06935955': None,
                        '06895000': None,
                        '06799315': '06799000',
                        '06881000': '06880800',
                        '06935755': None,
                        '06921590': None,
                        '06917060': '06917000',
                        '06211000': '06211500',
                        '06869950': None,
                        '06308500': '06326500',
                        '06093200': '06091700',
                        '06450500': '06449500',
                        '06909500': None,
                        '06834000': None}

In [10]:
import sys
import subprocess

# 1. Confirm which Python the kernel is using
print("Kernel Python:", sys.executable)

# 2. Force install into this exact kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx"])

# 3. Confirm it is now visible to this kernel
result = subprocess.check_output([sys.executable, "-m", "pip", "show", "python-pptx"])
print(result.decode())

Kernel Python: c:\Users\mroth\Comp 549\Flood-Forecasting\.venv\Scripts\python.exe


CalledProcessError: Command '['c:\\Users\\mroth\\Comp 549\\Flood-Forecasting\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'python-pptx']' returned non-zero exit status 1.

In [11]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import polars as pl
import numpy as np
import io
import os
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from scipy.signal import correlate
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

def _plot_gauge_comparison(
    df_pivot: pl.DataFrame,
    df_overlap: pl.DataFrame,
    primary_site_id: str,
    upstream_site_id: str,
    results: dict,
) -> plt.Figure:

    datetimes  = df_overlap["datetime"].to_list()
    primary_q  = df_overlap["primary"].to_numpy()
    upstream_q = df_overlap["upstream"].to_numpy()

    n_primary  = df_pivot["primary"].drop_nulls().len()
    n_upstream = df_pivot["upstream"].drop_nulls().len()
    n_overlap  = df_overlap.height

    color_primary  = "#1f77b4"
    color_upstream = "#ff7f0e"

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(
        f"Gauge Comparison:  {primary_site_id} (primary)  vs  {upstream_site_id} (upstream)",
        fontsize=13, fontweight="bold",
    )

    ax.plot(datetimes, primary_q,  color=color_primary,  linewidth=0.8,
            alpha=0.85, label=f"Primary ({primary_site_id})")
    ax.plot(datetimes, upstream_q, color=color_upstream, linewidth=0.8,
            alpha=0.85, label=f"Upstream ({upstream_site_id})")

    ax.set_ylabel("Streamflow (cfs)", fontsize=11)
    ax.set_xlabel("Date", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)

    date_range_days = (df_overlap["datetime"].max() - df_overlap["datetime"].min()).days
    if date_range_days <= 30:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    elif date_range_days <= 365:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    ax.annotate(
        f"n primary:  {n_primary:,}\nn upstream: {n_upstream:,}\nn overlap:  {n_overlap:,}",
        xy=(0.01, 0.03), xycoords="axes fraction",
        va="bottom", ha="left", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
    )
    ax.annotate(
        f"Lag:        {results['lag_hours']:.2f} hrs\n"
        f"m (scale):  {results['scale_m']:.4f}\n"
        f"b (other):  {results['intercept_b']:.2f} cfs\n"
        f"R²:         {results['r_squared']:.4f}\n"
        f"Pearson r:  {results['pearson_r']:.4f}",
        xy=(0.01, 0.97), xycoords="axes fraction",
        va="top", ha="left", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
    )

    plt.tight_layout()
    # plt.show()
    return fig

In [12]:
def analyze_gauge_relationship(
    parquet_path: str,
    primary_site_id: str,
    upstream_site_id: str,
    max_lag_hours: float = 72.0,
    timestep_minutes: int = 15,
    plot: bool = True,
) -> tuple[dict, pl.DataFrame, pl.DataFrame]:

    timesteps_per_hour = 60 // timestep_minutes
    max_lag_steps = int(max_lag_hours * timesteps_per_hour)

    df = (
        pl.scan_parquet(parquet_path)
        .filter(pl.col("site_id").is_in([primary_site_id, upstream_site_id]))
        .select(["site_id", "datetime", "streamflow_cfs"])
        .collect()
    )

    df_pivot = (
        df
        .pivot(index="datetime", on="site_id", values="streamflow_cfs")
        .sort("datetime")
        .rename({primary_site_id: "primary", upstream_site_id: "upstream"})
    )

    SENTINEL = -999_999
    df_pivot = df_pivot.with_columns([
        pl.when(pl.col("primary")  == SENTINEL).then(None).otherwise(pl.col("primary") ).alias("primary"),
        pl.when(pl.col("upstream") == SENTINEL).then(None).otherwise(pl.col("upstream")).alias("upstream"),
    ])

    n_primary  = df_pivot["primary"].drop_nulls().len()
    n_upstream = df_pivot["upstream"].drop_nulls().len()
    df_overlap = df_pivot.filter(
        pl.col("primary").is_not_null() & pl.col("upstream").is_not_null()
    )
    n_overlap = df_overlap.height

    print(f"Non-null streamflow counts")
    print(f"  Primary  ({primary_site_id}):  {n_primary:,}")
    print(f"  Upstream ({upstream_site_id}): {n_upstream:,}")
    print(f"  Both non-null (overlap):       {n_overlap:,}\n")

    if n_overlap < 2:
        raise ValueError("Not enough overlapping non-null values to compute correlation.")

    primary_vals  = df_overlap["primary"].to_numpy()
    upstream_vals = df_overlap["upstream"].to_numpy()

    p_centered = primary_vals  - primary_vals.mean()
    u_centered = upstream_vals - upstream_vals.mean()

    full_corr  = correlate(p_centered, u_centered, mode="full")
    lags       = np.arange(-len(u_centered) + 1, len(p_centered))

    valid_mask     = (lags >= 0) & (lags <= max_lag_steps)
    valid_lags     = lags[valid_mask]
    valid_corr     = full_corr[valid_mask]

    best_idx       = np.argmax(valid_corr)
    best_lag_steps = int(valid_lags[best_idx])
    best_lag_hours = best_lag_steps / timesteps_per_hour

    print(f"Cross-correlation best lag")
    print(f"  Lag steps:  {best_lag_steps} ({timestep_minutes}-min steps)")
    print(f"  Lag hours:  {best_lag_hours:.2f} hrs\n")

    if best_lag_steps > 0:
        upstream_shifted = upstream_vals[:-best_lag_steps]
        primary_aligned  = primary_vals[best_lag_steps:]
    else:
        upstream_shifted = upstream_vals
        primary_aligned  = primary_vals

    X = upstream_shifted.reshape(-1, 1)
    y = primary_aligned

    reg         = LinearRegression().fit(X, y)
    scale_m     = float(reg.coef_[0])
    intercept_b = float(reg.intercept_)
    r_squared   = float(reg.score(X, y))
    pearson_r, p_value = pearsonr(upstream_shifted, primary_aligned)

    print(f"Linear relationship:  primary = m * upstream + b")
    print(f"  m (scale factor):   {scale_m:.4f}  — {'attenuation' if scale_m < 1 else 'amplification'}")
    print(f"  b (other sources):  {intercept_b:.2f} cfs  — {'net gain' if intercept_b > 0 else 'net loss'} between gauges")
    print(f"  R²:                 {r_squared:.4f}")
    print(f"  Pearson r:          {pearson_r:.4f}  (p = {p_value:.2e})\n")

    results = {
        "lag_hours":    best_lag_hours,
        "lag_steps":    best_lag_steps,
        "scale_m":      scale_m,
        "intercept_b":  intercept_b,
        "r_squared":    r_squared,
        "pearson_r":    pearson_r,
        "n_primary":    n_primary,
        "n_upstream":   n_upstream,
        "n_overlap":    n_overlap,
    }

    if plot:
        fig = _plot_gauge_comparison(df_pivot, df_overlap, primary_site_id, upstream_site_id, results)
        plt.show()
        plt.close(fig)

    return results, df_pivot, df_overlap

In [13]:
import wandb
import os

api = wandb.Api()
artifact = api.artifact("flood-forecasting/raw-streamflow-15min:latest")
artifact_dir = artifact.download()

parquet_path = os.path.join(artifact_dir, "streamflow_15min", "*.parquet")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\mroth\.netrc.
wandb: Downloading large artifact 'raw-streamflow-15min:latest', 1467.48MB. 20 files...
wandb:   20 of 20 files downloaded.  
Done. 00:00:00.3 (5188.2MB/s)


In [ ]:
# results = analyze_gauge_relationship(
#     parquet_path     = parquet_path,
#     primary_site_id  = "06700000",
#     upstream_site_id = "06696980",
#     max_lag_hours    = 72,
#     timestep_minutes = 15,
# )

In [ ]:
# results = analyze_gauge_relationship(
#     parquet_path     = parquet_path,
#     primary_site_id  = "06820500",
#     upstream_site_id = "06819550",
#     max_lag_hours    = 72,
#     timestep_minutes = 15,
# )

In [17]:
def create_gauge_report_pptx(
    upstream_pair_dict: dict,
    output_path: str,
    parquet_path: str,
    max_lag_hours: float = 72.0,
    timestep_minutes: int = 15,
) -> str:
    """
    Creates a PowerPoint report with one slide per gauge pair.

    Parameters
    ----------
    upstream_pair_dict : {primary_site_id: upstream_site_id, ...}
                         upstream_site_id may be None if no match was found
    output_path        : Directory to save the .pptx file
    parquet_path       : Glob path to parquet files
    max_lag_hours      : Passed to analyze_gauge_relationship
    timestep_minutes   : Passed to analyze_gauge_relationship

    Returns
    -------
    str : Full path to the saved .pptx file
    """

    prs = Presentation()
    prs.slide_width  = Inches(13.33)
    prs.slide_height = Inches(7.5)

    blank_layout = prs.slide_layouts[6]

    for primary_id, upstream_id in upstream_pair_dict.items():

        slide = prs.slides.add_slide(blank_layout)

        # Title — always added regardless of whether upstream exists
        title_text = (
            f"{primary_id}  vs.  {upstream_id}"
            if upstream_id is not None
            else f"{primary_id}  —  No upstream match found"
        )

        title_box = slide.shapes.add_textbox(
            Inches(0.3), Inches(0.15),
            Inches(12.7), Inches(0.6),
        )
        tf = title_box.text_frame
        tf.word_wrap = False
        p = tf.paragraphs[0]
        p.text = title_text
        p.alignment = PP_ALIGN.CENTER
        run = p.runs[0]
        run.font.size = Pt(24)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0x1A, 0x1A, 0x2E)

        # Skip graph if no upstream gauge
        if upstream_id is None:
            print(f"Skipping graph for {primary_id} — no upstream match.")
            continue

        print(f"Processing: {primary_id} vs {upstream_id}")

        results, df_pivot, df_overlap = analyze_gauge_relationship(
            parquet_path     = parquet_path,
            primary_site_id  = primary_id,
            upstream_site_id = upstream_id,
            max_lag_hours    = max_lag_hours,
            timestep_minutes = timestep_minutes,
            plot             = False,
        )

        fig = _plot_gauge_comparison(df_pivot, df_overlap, primary_id, upstream_id, results)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
        buf.seek(0)
        plt.close(fig)

        slide.shapes.add_picture(
            buf,
            left   = Inches(0.15),
            top    = Inches(0.85),
            width  = Inches(13.0),
            height = Inches(6.5),
        )

    filename  = "gauge_pair_report.pptx"
    full_path = os.path.join(output_path, filename)
    prs.save(full_path)
    print(f"\nSaved: {full_path}")
    return full_path


In [22]:
output_path = r"C:\Users\mroth\OneDrive\Documents\Rice\COMP 549 DSCI 535 901 Capstone\Project"

pptx_path = create_gauge_report_pptx(
    upstream_pair_dict = upstream_pair_dict,
    output_path        = output_path,
    parquet_path       = parquet_path,
    max_lag_hours      = 72,
    timestep_minutes   = 15,
)

Processing: 06803500 vs 06803495
Non-null streamflow counts
  Primary  (06803500):  1,195,260
  Upstream (06803495): 1,047,636
  Both non-null (overlap):       992,262

Cross-correlation best lag
  Lag steps:  0 (15-min steps)
  Lag hours:  0.00 hrs

Linear relationship:  primary = m * upstream + b
  m (scale factor):   1.0548  — amplification
  b (other sources):  28.78 cfs  — net gain between gauges
  R²:                 0.9951
  Pearson r:          0.9975  (p = 0.00e+00)

Processing: 06446700 vs 06447500
Non-null streamflow counts
  Primary  (06446700):  429,075
  Upstream (06447500): 431,570
  Both non-null (overlap):       385,374

Cross-correlation best lag
  Lag steps:  0 (15-min steps)
  Lag hours:  0.00 hrs

Linear relationship:  primary = m * upstream + b
  m (scale factor):   0.8398  — attenuation
  b (other sources):  -3.02 cfs  — net loss between gauges
  R²:                 0.5679
  Pearson r:          0.7536  (p = 0.00e+00)

Processing: 06793000 vs 06790500
Non-null stre

In [ ]:
save_path = os.path.join(os.getcwd(), "upstream_pair_dict.json")

with open(save_path, "w") as f:
    json.dump(upstream_pair_dict, f, indent=4)

print(f"Saved to: {save_path}")

Saved to: c:\Users\mroth\Comp 549\Flood-Forecasting\notebooks\data_exploration\upstream_pair_dict.json
